# 04 · Features (3 places)

| Feature | Tool | For |
|---|---|---|
| sentences, lemmas | spaCy | BM25 keywords |
| embeddings | all-mpnet-base-v2 | meaning search |
| top 5 phrases | KeyBERT | place summary |
| sentiment | twitter-roberta | "cozy" vs. "not cozy" |

First run downloads ~1 GB of models.

In [1]:
import pandas as pd
import spacy
from keybert import KeyBERT
from sentence_transformers import SentenceTransformer
from transformers import pipeline

from vibecheck.filter_sf import REVIEWS_OUT, load_places
from vibecheck.report import write_section

places = load_places()
reviews = pd.read_parquet(REVIEWS_OUT)
places["atmosphere"] = places["misc"].map(lambda m: m.get("Atmosphere", []))

W1030 12:13:41.923000 69783 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


## Pick 3 places

Four Barrel Coffee (the slide example from notebook 01), plus the most-reviewed *Trending* bar and *Upscale* restaurant.

In [2]:
def pick(group: str, tag: str) -> pd.Series:
    """Most-reviewed place in `group` with Atmosphere `tag` (or without the tag if none has it)."""
    pool = places[places["group"] == group]
    tagged = pool[pool["atmosphere"].map(lambda tags: tag in tags)]
    return (tagged if len(tagged) else pool).sort_values("num_of_reviews", ascending=False).iloc[0]

cafe = places[places["name"] == "Four Barrel Coffee"].iloc[0]
examples = pd.DataFrame([cafe, pick("bar", "Trending"), pick("restaurant", "Upscale")])
examples[["name", "group", "atmosphere", "avg_rating", "num_of_reviews"]]

,name,group,atmosphere,avg_rating,num_of_reviews
4975,Four Barrel Coffee,cafe,"[Casual, Cozy]",4.5,2187
2600,The Beehive,bar,"[Casual, Cozy, Trending, Upscale]",4.5,347
5596,House of Prime Rib,restaurant,"[Cozy, Historic, Romantic, Upscale]",4.7,3913


## 1. spaCy sentences and lemmas

In [3]:
nlp = spacy.load("en_core_web_sm", exclude=["ner"])

def keywords(sent) -> list[str]:
    """Lowercased lemmas of content words: what BM25 will index."""
    return [t.lemma_.lower() for t in sent if t.is_alpha and not t.is_stop]

rows = []
for place in examples.itertuples():
    texts = reviews.loc[reviews["gmap_id"] == place.gmap_id, "text"]
    for doc in nlp.pipe(texts):
        rows += [{"gmap_id": place.gmap_id, "name": place.name, "sentence": s.text.strip(), "lemmas": keywords(s)}
                 for s in doc.sents if s.text.strip()]
sentences = pd.DataFrame(rows)
print(f"{len(sentences):,} sentences from {len(examples)} places")
sentences.head(5)

469 sentences from 3 places


,gmap_id,name,sentence,lemmas
0,0x808f7e218f9bf6ff:0xd739722d19b32b9b,Four Barrel Coffee,Great quality coffee.,"[great, quality, coffee]"
1,0x808f7e218f9bf6ff:0xd739722d19b32b9b,Four Barrel Coffee,One of the baristas likes to call every custom...,"[barista, like, customer, buddy, find, condesc..."
2,0x808f7e218f9bf6ff:0xd739722d19b32b9b,Four Barrel Coffee,Good coffee flavor but the oatmilk tastes so s...,"[good, coffee, flavor, oatmilk, taste, sour, c..."
3,0x808f7e218f9bf6ff:0xd739722d19b32b9b,Four Barrel Coffee,Maybe consider changing from minor figures to ...,"[maybe, consider, change, minor, figure]"
4,0x808f7e218f9bf6ff:0xd739722d19b32b9b,Four Barrel Coffee,This is a great place to grab a cup of coffee.,"[great, place, grab, cup, coffee]"


## 2. Embeddings

In [4]:
encoder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
embeddings = encoder.encode(sentences["sentence"].tolist(), batch_size=64, show_progress_bar=True,
                            normalize_embeddings=True)
print("embeddings shape:", embeddings.shape)  # (sentences, 768)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (469, 768)


## 3. KeyBERT phrases

Candidates are adjective + noun phrases from spaCy noun chunks ("comfortable seating"). Plain n-grams over
joined reviews gave junk like "bay area barrel". A small seed nudges KeyBERT toward the vibe, and MMR
keeps the five phrases different.

In [5]:
kw_model = KeyBERT(model=encoder)
VIBE_SEED = ["atmosphere", "vibe", "ambiance"]

def candidate_phrases(texts: pd.Series) -> list[str]:
    """Adjective + noun phrases (1-3 words) from spaCy noun chunks."""
    found = set()
    for doc in nlp.pipe(texts):
        for chunk in doc.noun_chunks:
            words = [t for t in chunk if not (t.is_stop or t.is_punct or t.pos_ in ("DET", "PRON", "NUM"))]
            if 1 <= len(words) <= 3 and any(t.pos_ == "ADJ" for t in words):
                found.add(" ".join(t.lower_ for t in words))
    return sorted(found)

def vibe_phrases(texts: pd.Series, n: int = 5) -> list[str]:
    """Top `n` diverse vibe phrases for one place's reviews."""
    found = kw_model.extract_keywords(" ".join(texts), candidates=candidate_phrases(texts),
                                      keyphrase_ngram_range=(1, 3), seed_keywords=VIBE_SEED,
                                      use_mmr=True, diversity=0.5, top_n=n)
    return [phrase for phrase, _ in found]

phrases = {p.gmap_id: vibe_phrases(reviews.loc[reviews["gmap_id"] == p.gmap_id, "text"]) for p in examples.itertuples()}
phrases

{'0x808f7e218f9bf6ff:0xd739722d19b32b9b': ['best coffee shop',
  'great ambiance',
  'friendly staff',
  'negative reviews',
  'average price'],
 '0x808f7e3da9be2791:0x5f5396b84ee4d5c0': ['beautiful cocktail bar',
  'indoor seating',
  'beehive drink',
  'playful new american',
  'attentive servers'],
 '0x808580ea69dcae83:0x3ef25abd292344d9': ['fantastic food',
  'generally credible restaurant',
  'quality prime rib',
  'old school atmosphere',
  'annual birthday visit']}

## 4. Sentiment

In [6]:
sentiment = pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                     truncation=True, max_length=128)
sentences["sentiment"] = [r["label"] for r in sentiment(sentences["sentence"].tolist(), batch_size=32)]
sentences[["name", "sentence", "sentiment"]].sample(8, random_state=0)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,name,sentence,sentiment
310,House of Prime Rib,Very hard to get a reservation.,negative
230,The Beehive,"If the gold on the glasses doesnt awll you, th...",neutral
21,Four Barrel Coffee,"Great coffee, cool spot.",positive
289,House of Prime Rib,Huge portions yummy slides and all the fixings.,positive
238,The Beehive,"The decor of the place was amazing, Bartenders...",positive
179,The Beehive,Do not go here.,negative
132,The Beehive,"I still like this place a lot, but preferred t...",positive
135,The Beehive,Soooo delicious,positive


## 5. Table for the slides

In [7]:
shares = sentences.groupby("gmap_id")["sentiment"].value_counts(normalize=True).unstack(fill_value=0)
summary = pd.DataFrame(
    {
        "place": examples["name"].values,
        "Google Atmosphere": examples["atmosphere"].map(", ".join).values,
        "sentences": sentences.groupby("gmap_id").size().reindex(examples["gmap_id"]).values,
        "top 5 vibe phrases (KeyBERT)": [", ".join(phrases[g]) for g in examples["gmap_id"]],
        "% positive": (100 * shares.reindex(examples["gmap_id"])["positive"]).round(0).values,
        "% negative": (100 * shares.reindex(examples["gmap_id"])["negative"]).round(0).values,
    }
)
md = f"""## Features prototype (notebook 04)

Embeddings: {embeddings.shape[0]:,} sentences x {embeddings.shape[1]} dims (all-mpnet-base-v2).

{summary.to_markdown(index=False)}
"""
write_section("features", md)
summary

,place,Google Atmosphere,sentences,top 5 vibe phrases (KeyBERT),% positive,% negative
0,Four Barrel Coffee,"Casual, Cozy",129,"best coffee shop, great ambiance, friendly sta...",74.0,12.0
1,The Beehive,"Casual, Cozy, Trending, Upscale",142,"beautiful cocktail bar, indoor seating, beehiv...",71.0,8.0
2,House of Prime Rib,"Cozy, Historic, Romantic, Upscale",198,"fantastic food, generally credible restaurant,...",63.0,12.0
